In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/train.csv")

# convert values
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%d/%m/%Y")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# totals
total_sales = df["Sales"].sum()
total_per_year = df.groupby(df["Order Date"].dt.to_period("Y"))["Sales"].sum().reset_index()

# calculate growth over years
total_per_year["Growth"] = total_per_year["Sales"].diff().fillna(0)
total_per_year["Percent Growth"] = total_per_year["Sales"].pct_change().fillna(0) * 100

total_per_year

In [ ]:
# plot sales by time
sales_by_time = df.groupby(df["Order Date"].dt.to_period("M"))["Sales"].sum()

rolling_mean_12 = sales_by_time.rolling(12).mean()
sales_by_time.plot(label="Monthly Sales")
rolling_mean_12.plot(label="12-Month Average")

plt.legend()

In [ ]:
# average sales by month of the year
avg_monthly_sales = (
    df.groupby(df["Order Date"].dt.to_period("M"))["Sales"]
    .sum()
    .groupby(lambda x: x.month)
    .mean()
)
# correct indexes
avg_monthly_sales.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

avg_monthly_sales.plot(kind="bar")

In [ ]:
# sales by category
sales_by_category = df.groupby("Category")["Sales"].sum().sort_values(ascending=False).reset_index()
sales_by_category["Share"] = sales_by_category["Sales"] / df["Sales"].sum() * 100

ax = sales_by_category.plot(
    kind="barh",
    x="Category",
    y="Share",
    legend=False
)

bars = ax.barh(
    sales_by_category["Category"],
    sales_by_category["Share"]
)

for bar, sales in zip(bars, sales_by_category["Sales"]):
    width = bar.get_width()
    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}% (${sales:,.0f})",
        va="center"
    )

plt.xlabel("Sales Share (%)")
plt.title("Sales Share by Category")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# sales by region
sales_by_region = df.groupby("Region")["Sales"].sum().reset_index()
sales_by_region["Share"] = sales_by_region["Sales"] / df["Sales"].sum() * 100

ax = sales_by_region.plot(
    kind="barh",
    x="Region",
    y="Share",
    legend=False
)

bars = ax.barh(
    sales_by_region["Region"],
    sales_by_region["Share"]
)

for bar, sales in zip(bars, sales_by_region["Sales"]):
    width = bar.get_width()
    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}% (${sales:,.0f})",
        va="center"
    )

plt.xlabel("Sales Share (%)")
plt.title("Sales Share by Region")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# sales by customer segment
sales_by_customer_segment = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False).reset_index()
sales_by_customer_segment["Share"] = sales_by_customer_segment["Sales"] / df["Sales"].sum() * 100

ax = sales_by_customer_segment.plot(
    kind="barh",
    x="Segment",
    y="Share",
    legend=False
)

bars = ax.barh(
    sales_by_customer_segment["Segment"],
    sales_by_customer_segment["Share"]
)

for bar, sales in zip(bars, sales_by_customer_segment["Sales"]):
    width = bar.get_width()
    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}% (${sales:,.0f})",
        va="center"
    )

plt.xlabel("Sales Share (%)")
plt.title("Sales Share by Customer Segment")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# top 10 products by revenue
top_revenue_products = (
    df.groupby(["Product Name", "Category", "Sub-Category"])["Sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .head(10)
)
top_revenue_products["Share"] = top_revenue_products["Sales"] / df["Sales"].sum() * 100

top_revenue_products

In [ ]:
# top 10 sub-categories
top_revenue_subcategories = (
    df.groupby("Sub-Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .head(10)
)

top_revenue_subcategories["Share"] = top_revenue_subcategories["Sales"] / df["Sales"].sum() * 100

top_revenue_subcategories

In [ ]:
# revenue by weekday
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
revenue_by_weekday = (
    df.groupby(df["Order Date"].dt.day_name())["Sales"]
    .sum()
    .reindex(weekday_order)
)

revenue_by_weekday.plot(kind="bar", title="Revenue by Weekday")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.show()